In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [3]:
import pandas as pd
data=pd.read_csv("./data/demo2.csv",encoding="gbk").sample(frac=1)
data.head(5)

,label,review
4353,1,房间里有冰箱，十分方便。但是离开房间后必须将卡插上，否则没电。正在部分改造，楼道有油漆味。房...
1491,1,总体感觉很不错，酒店的地理位置很好，出行很方便，而且对面就是市政府，景观效果很好，房间面积也...
2286,1,五一节去的，住的是家庭房，别的倒还可以，不过不知道为什么房间里蚊子特别多，全家被蚊子咬的伤痕...
5859,0,房间在这个价位来说可以接受，卫生间很大！但是隔音非常差，什么都听得到。最差的是早餐，简直是恐...
6642,0,"据说这是泰山上最高级的酒店,3星的,但自从我们一进酒店大门就发现,这个在山下只能算是1星的标..."


In [4]:
x_train=data[:5000]
x_test=data[5000:]
y_train=x_train["label"]
y_test=x_test["label"]
del x_train["label"]
del x_test["label"]

### 简单跑一个模型

In [5]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *
from easymlops.nlp.preprocessing import *
from easymlops.nlp.representation import *
from easymlops.table.classification import *
from easymlops.table.ensemble import *

In [6]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(ExtractChineseWords())\
  .pipe(ExtractJieBaWords())\
  .pipe(Parallel([LsiTopicModel(num_topics=128),Word2VecModel(embedding_size=128)]))\
  .pipe(Normalizer())\
  .pipe(LGBMClassification(y=y_train,native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},prefix="lgbm"))

table.fit(x_train)

### 挂载

In [7]:
from easymlops.table.storage import FaissStorage

In [8]:
table[-2].set_branch_pipe(FaissStorage(raw_data=x_train["review"].values.tolist(),storage_path="./faiss_storage",
                                       create_index_param="IVF100,PQ16",create_index_measure="METRIC_INNER_PRODUCT"))

In [9]:
table[-2].get_branch_pipe(0).fit(table.transform(x_train,run_to_layer=-2))

<FaissStorage(<class 'easymlops.table.storage.faiss_storage.FaissStorage'>, started 3308)>

### 查询历史相似数据

In [10]:
input_data={"review":"地方偏僻，而且部根本是人服的，前天去住的候，空都有，酒店借口，已了，但是一都有，了省，搞的，..."}
table[-2].get_branch_pipe(0).search(table.transform_single(input_data,run_to_layer=-2))

,similarity_top_1_raw_data,similarity_top_2_raw_data,similarity_top_3_raw_data,similarity_top_1_measure,similarity_top_2_measure,similarity_top_3_measure
0,我十一期间入住到瑞和园，先把最坏的缺点透露给驴友（免得还没看到最后就翻页了）：隔音效果极差！...,就住了两天，第一天晚上就发现了一只大小强。第二天下午回来让服务员喷了药，早上起来发现了一只殉...,两间房只给两条毛巾，加一条毛巾要收费5元，太过份了吧？？？,64.660576,59.948013,58.671753


### 模拟线上数据流

In [11]:
i=0
for record in tqdm(x_test[:10].to_dict("record")):
    table.transform_single(record,storage_base_dict={"key":i,"row_data":record.get("review")},add_index=True)
    i+=1

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 34.79it/s]


In [12]:
input_data=x_test[:10].to_dict("record")[9]
input_data

{'review': '交通便利，房间很整洁，价格也合理，服务很不错，工作人员态度也很好，比较满意，唯一不足的就是离4星似乎还差一点！'}

In [13]:
table[-2].get_branch_pipe(0).search(table.transform_single(input_data,run_to_layer=-2))

,similarity_top_1_raw_data,similarity_top_2_raw_data,similarity_top_3_raw_data,similarity_top_1_measure,similarity_top_2_measure,similarity_top_3_measure
0,环境地理位置很好，交通方便。房间舒适，服务很好,"房间很大,设施不错,交通方便,环境很好,很安静.另外服务和早餐也不错.",酒店很不错，硬件很好。自助餐厅早、中、晚都有很方便。性价比很高！,383.212524,381.70459,376.879303


### 异步保存
存储模块强制使用异步的方式存储数据，通常只消耗少量transform_single的时间(获取当前时间以及字典中的数据，并放到队列中异步存储)

In [14]:
from easymlops.table.callback import check_transform_function_pipeline
table.callback(check_transform_function_pipeline,x_test[:10])

(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.ExtractChineseWords'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.ExtractJieBaWords'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.ensemble.Parallel'>) module check [transform] complete,speed:[21.15ms]/it,cpu:[25%],memory:[20K]
(<class 'easymlops.table.preprocessing.onevar_operation.Normalizer'>) module check [transform] complete,speed:[3.13ms]/it,cpu:[23%],memory:[0K]
(<class 'easymlops.table.classification.LGBMClassification'>) module check [transform] complete,speed:[3.13ms]/it,cpu:[81%],memory:[0K]


### 持久化测试

In [15]:
table[-2].get_branch_pipe(0).save_index_raw_data()

In [16]:
table.save("ml.pkl")

In [17]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(ExtractChineseWords())\
  .pipe(ExtractJieBaWords())\
  .pipe(Parallel([LsiTopicModel(num_topics=128),Word2VecModel(embedding_size=128)]))\
  .pipe(Normalizer())\
  .pipe(LGBMClassification(y=y_train,native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},prefix="lgbm"))

table[-2].set_branch_pipe(FaissStorage())
table.load("ml.pkl")

In [18]:
table[-2].get_branch_pipe(0).load_index_raw_data()

In [19]:
input_data=x_test[:10].to_dict("record")[9]
input_data

{'review': '交通便利，房间很整洁，价格也合理，服务很不错，工作人员态度也很好，比较满意，唯一不足的就是离4星似乎还差一点！'}

In [20]:
table[-2].get_branch_pipe(0).search(table.transform_single(input_data,run_to_layer=-2))

,similarity_top_1_raw_data,similarity_top_2_raw_data,similarity_top_3_raw_data,similarity_top_1_measure,similarity_top_2_measure,similarity_top_3_measure
0,环境地理位置很好，交通方便。房间舒适，服务很好,"房间很大,设施不错,交通方便,环境很好,很安静.另外服务和早餐也不错.",酒店很不错，硬件很好。自助餐厅早、中、晚都有很方便。性价比很高！,383.212524,381.70459,376.879303
